# 02 — ByteTrack setup (tracking, CPU)

ByteTrack is an **algorithm**, not a huge neural net.

This notebook:

1. Saves / verifies `models/tracker/bytetrack.yaml`
2. Installs tracking dependencies (`lapx`, `scipy`, `filterpy`)
3. Optionally clones reference ByteTrack repo for study
4. Confirms the **in-project CPU tracker** at `src/tracking/bytetrack.py`

> The pipeline already includes a lightweight ByteTrack-style tracker.  
> You do **not** need GPU BoT-SORT.


In [1]:
from pathlib import Path
import sys

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "main.py").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root from the current notebook location.")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))
TRACK_DIR = ROOT / "models" / "tracker"
TRACK_DIR.mkdir(parents=True, exist_ok=True)
print(ROOT)


/Users/macbookpro/Desktop/person-face-events


In [ ]:
# %pip install lapx scipy filterpy numpy


In [2]:
import yaml
from pathlib import Path

cfg = {
    "tracker_type": "bytetrack",
    "track_high_thresh": 0.5,
    "track_low_thresh": 0.1,
    "new_track_thresh": 0.6,
    "track_buffer": 30,
    "match_thresh": 0.8,
    "fuse_score": True,
}
path = Path("..").resolve() / "models" / "tracker" / "bytetrack.yaml"
path.write_text(yaml.dump(cfg, default_flow_style=False), encoding="utf-8")
print("Wrote", path)
print(path.read_text())


Wrote /Users/macbookpro/Desktop/person-face-events/models/tracker/bytetrack.yaml
fuse_score: true
match_thresh: 0.8
new_track_thresh: 0.6
track_buffer: 30
track_high_thresh: 0.5
track_low_thresh: 0.1
tracker_type: bytetrack



In [ ]:
# Optional: clone official ByteTrack reference (algorithm source, not required at runtime)
from pathlib import Path
import subprocess

ref = Path("..").resolve() / "models" / "tracker" / "ByteTrack-ref"
if not ref.exists():
    print("Cloning ByteTrack reference (shallow)...")
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/ifzhang/ByteTrack.git",
        str(ref)
    ], check=False)
else:
    print("Already present:", ref)

print("Runtime tracker uses: src/tracking/bytetrack.py (CPU IoU ByteTrack-style)")


Cloning ByteTrack reference (shallow)...


Cloning into '/Users/macbookpro/Desktop/person-face-events/models/tracker/ByteTrack-ref'...


In [ ]:
# Smoke test in-project tracker
import sys
from pathlib import Path
import numpy as np
sys.path.insert(0, str(ROOT))

from src.detection.person_yolo import Detection
from src.tracking.bytetrack import ByteTracker

tr = ByteTracker()
dets = [
    Detection(xyxy=np.array([100, 100, 200, 300], dtype=float), conf=0.9),
    Detection(xyxy=np.array([400, 120, 500, 320], dtype=float), conf=0.8),
]
tracks = tr.update(dets)
print("Tracks:", [(t.track_id, t.centroid) for t in tracks])

# move boxes slightly — IDs should stay stable
dets2 = [
    Detection(xyxy=np.array([105, 102, 205, 302], dtype=float), conf=0.88),
    Detection(xyxy=np.array([398, 118, 498, 318], dtype=float), conf=0.81),
]
tracks2 = tr.update(dets2)
print("Tracks2:", [(t.track_id, t.centroid) for t in tracks2])
print("✓ ByteTrack CPU tracker OK")


## Ultralytics built-in ByteTrack (optional)

If you prefer Ultralytics tracker API:

```python
from ultralytics import YOLO
model = YOLO("models/yolo/yolo11n.pt")
# results = model.track(source=frame, tracker="bytetrack.yaml", classes=[0], device="cpu")
```

Our `main.py` uses the custom `ByteTracker` so entry/exit state (`prev_side`, `person_name`) lives on the track object.
